In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

CATALOG = "olist_dev"
S, G = f"{CATALOG}.silver", f"{CATALOG}.gold"
DIM = f"{G}.dim_customer_scd2"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM} (
  customer_sk BIGINT,
  customer_unique_id STRING,
  zip_prefix STRING,
  city STRING,
  state STRING,
  row_hash STRING,
  effective_from TIMESTAMP,
  effective_to TIMESTAMP,
  is_current BOOLEAN
)
""")

DataFrame[]

In [0]:
from pyspark.sql.window import Window

def build_source():
    c = spark.table(f"{S}.customers")
    o = spark.table(f"{S}.orders").select("customer_id", "order_purchase_timestamp")
    w = Window.partitionBy("customer_unique_id").orderBy(F.col("order_purchase_timestamp").desc_nulls_last())
    src = (c.join(o, "customer_id", "left")
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
        .select("customer_unique_id",
                F.col("customer_zip_code_prefix").alias("zip_prefix"),
                F.col("customer_city").alias("city"),
                F.col("customer_state").alias("state"))
        .withColumn("row_hash", F.sha2(F.concat_ws("||",
            F.coalesce("zip_prefix", F.lit("")),
            F.coalesce("city", F.lit("")),
            F.coalesce("state", F.lit(""))), 256)))
    return src

src = build_source()
print("Source customers:", src.count())   # ~96,096

Source customers: 96096


In [0]:
def scd2_merge(src, load_ts):
    ts = load_ts.strftime("%Y-%m-%d %H:%M:%S")
    current = spark.table(DIM).filter("is_current = true")

    # 1. Jin customers ka hash badla, unki "naya version" wali row
    changed = (src.alias("s")
        .join(current.alias("t"), F.col("s.customer_unique_id") == F.col("t.customer_unique_id"))
        .filter(F.col("s.row_hash") != F.col("t.row_hash"))
        .select("s.*"))

    # 2. Staged = changed (merge_key NULL, taaki insert ho) + saari source rows (merge_key = id)
    staged = (changed.withColumn("merge_key", F.lit(None).cast("string"))
        .unionByName(src.withColumn("merge_key", F.col("customer_unique_id")))
        .withColumn("customer_sk", F.xxhash64(F.col("customer_unique_id"), F.lit(ts), F.col("row_hash"))))

    (DeltaTable.forName(spark, DIM).alias("t")
        .merge(staged.alias("s"),
               "t.customer_unique_id = s.merge_key AND t.is_current = true")
        # Match + hash alag: purani row band karo
        .whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash",
            set={"is_current": "false", "effective_to": f"timestamp'{ts}'"})
        # Match nahi: naya customer ya changed customer ka naya version
        .whenNotMatchedInsert(values={
            "customer_sk": "s.customer_sk",
            "customer_unique_id": "s.customer_unique_id",
            "zip_prefix": "s.zip_prefix",
            "city": "s.city",
            "state": "s.state",
            "row_hash": "s.row_hash",
            "effective_from": f"timestamp'{ts}'",
            "effective_to": "timestamp'9999-12-31 00:00:00'",
            "is_current": "true"})
        .execute())
    print("Dim total rows:", spark.table(DIM).count(),
          "| current:", spark.table(DIM).filter("is_current").count())

In [0]:
scd2_merge(src, datetime(2026, 1, 1))

Dim total rows: 96772 | current: 96096


In [0]:
src_changed = (build_source()
    .withColumn("_chg", F.pmod(F.xxhash64("customer_unique_id"), F.lit(300)) == 0)
    .withColumn("city", F.when(F.col("_chg"), F.lit("novo_city")).otherwise(F.col("city")))
    .withColumn("state", F.when(F.col("_chg"), F.lit("XX")).otherwise(F.col("state")))
    .withColumn("row_hash", F.sha2(F.concat_ws("||",
        F.coalesce("zip_prefix", F.lit("")),
        F.coalesce("city", F.lit("")),
        F.coalesce("state", F.lit(""))), 256))
    .drop("_chg"))

print("Changed rows in source:", src_changed.filter("city = 'novo_city'").count())

scd2_merge(src_changed, datetime(2026, 6, 1))

Changed rows in source: 338
Dim total rows: 97110 | current: 96096


In [0]:
%sql
SELECT customer_unique_id, city, state, effective_from, effective_to, is_current
FROM olist_dev.gold.dim_customer_scd2
WHERE customer_unique_id IN (
  SELECT customer_unique_id FROM olist_dev.gold.dim_customer_scd2
  GROUP BY customer_unique_id HAVING COUNT(*) > 1 LIMIT 3)
ORDER BY customer_unique_id, effective_from;

customer_unique_id,city,state,effective_from,effective_to,is_current
2f3d09b69266cbab293052a3f0701f61,rio de janeiro,RJ,2026-01-01T00:00:00.000Z,2026-06-01T00:00:00.000Z,false
2f3d09b69266cbab293052a3f0701f61,rio de janeiro,RJ,2026-01-01T00:00:00.000Z,2026-06-01T00:00:00.000Z,false
2f3d09b69266cbab293052a3f0701f61,novo_city,XX,2026-06-01T00:00:00.000Z,2026-01-01T00:00:00.000Z,false
2f3d09b69266cbab293052a3f0701f61,novo_city,XX,2026-06-01T00:00:00.000Z,9999-12-31T00:00:00.000Z,true
4311c3444d110af4d6e59fa7e02c6876,guaruja,SP,2026-01-01T00:00:00.000Z,2026-06-01T00:00:00.000Z,false
4311c3444d110af4d6e59fa7e02c6876,guaruja,SP,2026-01-01T00:00:00.000Z,2026-06-01T00:00:00.000Z,false
4311c3444d110af4d6e59fa7e02c6876,novo_city,XX,2026-06-01T00:00:00.000Z,9999-12-31T00:00:00.000Z,true
4311c3444d110af4d6e59fa7e02c6876,novo_city,XX,2026-06-01T00:00:00.000Z,2026-01-01T00:00:00.000Z,false
f996e11abe51257c00e339b7003b988c,caxias do sul,RS,2026-01-01T00:00:00.000Z,2026-06-01T00:00:00.000Z,false
f996e11abe51257c00e339b7003b988c,caxias do sul,RS,2026-01-01T00:00:00.000Z,2026-06-01T00:00:00.000Z,false


In [0]:
scd2_merge(src_changed, datetime(2026, 6, 2))

Dim total rows: 97110 | current: 96096


In [0]:
d = spark.table(DIM)

# 1. Har customer ki exactly 1 current row
bad = d.filter("is_current").groupBy("customer_unique_id").count().filter("count <> 1").count()
print("Customers with != 1 current row:", bad)          # 0

# 2. Expired rows ka effective_to kabhi 9999 nahi hona chahiye
print("Bad expired rows:", d.filter("is_current = false AND effective_to > '9000-01-01'").count())  # 0

# 3. Current rows ka effective_to 9999 hona chahiye
print("Bad current rows:", d.filter("is_current = true AND effective_to < '9000-01-01'").count())   # 0

Customers with != 1 current row: 0
Bad expired rows: 0
Bad current rows: 0


In [0]:
%sql
SELECT customer_unique_id, city, state
FROM olist_dev.gold.dim_customer_scd2
WHERE customer_unique_id IN (SELECT customer_unique_id FROM olist_dev.gold.dim_customer_scd2 WHERE city = 'novo_city' LIMIT 3)
  AND timestamp'2026-05-01 00:00:00' >= effective_from
  AND timestamp'2026-05-01 00:00:00' <  effective_to;

customer_unique_id,city,state
04bb7cdad7b0b034c528545519ac8096,santo andre,SP
1495ea1c194653b4e0f83cfc66918185,conceicao do rio verde,MG
f996e11abe51257c00e339b7003b988c,caxias do sul,RS
04bb7cdad7b0b034c528545519ac8096,santo andre,SP
f996e11abe51257c00e339b7003b988c,caxias do sul,RS
1495ea1c194653b4e0f83cfc66918185,conceicao do rio verde,MG


In [0]:
from datetime import datetime
scd2_merge(build_source(), datetime.utcnow())
dbutils.notebook.exit("scd2_ok")